In [5]:
import os

os.makedirs("output", exist_ok=True)

# **Part 1**

In [ ]:
import pandas as pd


def clean_and_import():

    # Problem 1.1
    eurostat = pd.read_excel("data/eurostat.xlsx", sheet_name="Sheet 1", header=9, na_values=[":"])
    rows_before = eurostat.shape[0]

    # Selects the rows with values. 
    # I'm using iloc instead of skipfooter so I can record the number of rows imported.
    eurostat = eurostat.iloc[1:-5]
    rows_now = eurostat.shape[0]
    rows_removed = rows_before-rows_now
    
    # Selects all rows, but only columns where the column name DOESN'T start with Unnamed.
    eurostat = eurostat.iloc[:, ~eurostat.columns.astype(str).str.startswith("Unnamed")]
    print(f"{'Rows before:':<15}{rows_before:<8}")
    print(f"{'Rows now:':<15}{rows_now:<8}")
    print(f"{'Rows removed:':<15}{rows_removed:<8}")
        
    # Problem 1.2
    eurostat = eurostat.rename(columns={"TIME": "Country"})

    countries = eurostat.iloc[2:].reset_index(drop=True)
    aggregates = eurostat.iloc[0:2].reset_index(drop=True)

    return countries, aggregates


countries, _ = clean_and_import()

# Problem 1.4
rows_missing = countries.iloc[:, 1:].isna().sum()

# Most countries have data up to 2022. 2023 stands out as only three countries have data that year.
# The data for the UK is missing from 2020. A possible explaination for the missing data, might be 
# their exit from the united kingdom.

#print(rows_missing)

# Part 2

In [140]:
import numpy as np

countries, _ = clean_and_import()

def pct_change(old, new):
    if pd.isna(old) or pd.isna(new):
        return np.nan
    if old <= 0:
        raise ValueError(f"Old is {old}. Expected a value larger than 0")
    else:
        return (new/old-1)*100


# Problem 1.2
countries["pct_change"] = np.nan
for i in range(0, countries.shape[0]):
    row = countries.iloc[i]
    name, old, new = row["Country"], row["2002"], row["2022"]
    countries.at[i, "pct_change"] = pct_change(old, new)

# Problem 1.3

countries["pct_change_vectorized"] = (countries["2022"]/countries["2002"]-1)*100
lv = np.isclose(
    countries["pct_change_vectorized"],
    countries["pct_change"],
    equal_nan=True,
)

if lv.all():
    print("\nVectorized and loop columns yield the same percentage change")

Rows before:   49      
Rows now:      43      
Rows removed:  6       

Vectorized and loop columns yield the same percentage change


In [137]:
# tests for pct_change

for old, new in [(100, 150), (200, 150), (80, 80), (100, np.nan), (np.nan, 100)]:
    print(f"pct_change({old}, {new}) = {pct_change(old, new)}")

# Cases that must raise
for old, new in [(0, 50), (-10, 50)]:
    try:
        pct_change(old, new)
    except ValueError as e:
        print(f"pct_change({old}, {new}) raised ValueError: {e}")

pct_change(100, 150) = 50.0
pct_change(200, 150) = -25.0
pct_change(80, 80) = 0.0
pct_change(100, nan) = nan
pct_change(nan, 100) = nan
pct_change(0, 50) raised ValueError: Old is 0. Expected a value larger than 0
pct_change(-10, 50) raised ValueError: Old is -10. Expected a value larger than 0


# Part 3

In [ ]:
# Problem 3.1

if "pct_change" not in countries.columns:
    countries, _ = clean_and_import()
    countries["pct_change"] = (countries["2022"] / countries["2002"] - 1) * 100

change = countries["pct_change_vectorized"]

conditions = [change.isna(), change >= 20, (change >= 5) & (change < 20), (change > -5) & (change < 5), change <= -5]
labels = ["no data", "strong growth", "growth", "stable", "decline"]
countries["trend"] = np.select(conditions, labels, default="no data")

# Problem 3.2
# Default sorting is ascending (lowest values first)
top = list(zip(
    countries.nlargest(5, "pct_change")["Country"], 
    countries.nlargest(5, "pct_change")["pct_change"].round(1)))

bottom = list(zip(
    countries.nsmallest(5, "pct_change")["Country"], 
    countries.nsmallest(5, "pct_change")["pct_change"].round(1)
))

print("Highest:", top)
print("Lowest: ", bottom)

# Problem 3.3

Highest: [('Türkiye', 178.4), ('Malta', 62.7), ('Lithuania', 51.3), ('Albania', 45.1), ('Poland', 43.8)]
Lowest:  [('Kosovo*', -81.1), ('Germany', -6.9), ('Sweden', -5.6), ('Finland', -2.9), ('Denmark', -2.4)]
